# Code Review — Phases 1 & 2 of `agentic-sdlc`

*A walkthrough, as if we were reviewing the diff together. I'll show you what changed, run the
functions live, and explain how each piece works.*

**What we're reviewing**

| PR | Phase | What it does |
|----|-------|--------------|
| **#8** | 1 | `ship_issue` stops owning isolation + runs non-interactively; load-bearing bash extracted to **tested scripts** |
| **#9** | 2 | Two-tier **knowledge model** (invariants + constitution) feeding the compounding loop |

PR #9 is stacked on #8, so this worktree contains **both** phases — we can demo everything from here.

In [1]:
# Setup — locate the repo root (works wherever you launch this from) so the
# bash cells can find scripts/, KNOWLEDGE.md, etc. No hardcoded paths.
import os
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.exists(os.path.join(d, "scripts", "lib.sh")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError("Could not find repo root (scripts/lib.sh). "
                               "Launch Jupyter from the repo or the demo/ folder.")
        d = parent
ROOT = find_root()
os.environ["ROOT"] = ROOT
print("Repo root:", ROOT)

Repo root: /home/matt/projects/agentic-sdlc/.claude/worktrees/phase-2


In [2]:
%%bash
cd "$ROOT"
echo "branch: $(git branch --show-current 2>/dev/null || echo '(unknown)')"
# Tolerant diff: try local then origin/ refs; never error if a branch is absent.
show_diff () {  # <base> <head>
  for b in "$1" "origin/$1"; do for h in "$2" "origin/$2"; do
    if git rev-parse --verify -q "$b" >/dev/null && git rev-parse --verify -q "$h" >/dev/null; then
      git diff --stat "$b" "$h"; return 0
    fi
  done; done
  echo "(branches $1..$2 not present locally — see PRs #8 / #9 on GitHub)"
}
echo; echo "=== Phase 1 (main -> worktree-phase-1) ==="; show_diff main worktree-phase-1
echo; echo "=== Phase 2 (worktree-phase-1 -> worktree-phase-2) ==="; show_diff worktree-phase-1 worktree-phase-2

branch: worktree-phase-2



=== Phase 1 (main -> worktree-phase-1) ===


 .claude-plugin/plugin.json        |   2 +-
 MODELS.md                         |  39 +++++++++++
 RE

ADME.md                         |  13 ++++
 commands/ship_issue.md            | 133 +++++++++++++++-

----------------------
 scripts/lib.sh                    |  72 +++++++++++++++++++++
 scripts/prune

...


=== Phase 2 (worktree-phase-1 -> worktree-phase-2) ===


 .claude-plugin/plugin.json |  4 +--
 KNOWLEDGE.md               | 74 ++++++++++++++++++++++++++++++

++++++++++++++++
 README.md                  |  8 ++++-
 agents/code-simplifier.md  |  6 ++--
 comma

nds/ship_issue.md     | 58 +++++++++++++++++++++++-------------
 templates/constitution.md  | 46 +++

+++++++++++++++++++++++++
 6 files changed, 171 insertions(+), 25 deletions(-)


---
# Part 1 — Phase 1: tested scripts + non-interactive `ship_issue`

The headline change: the brittle inline bash that used to live *inside* `ship_issue.md` (label
verification, review polling, worktree cleanup) is now **versioned, testable scripts**. The
**pure logic** (no `gh`/`git`) is factored into `lib.sh` so we can unit-test it with plain bash.

### 1.1 — The scripts at a glance

In [3]:
%%bash
cd "$ROOT"
ls -1 scripts/ scripts/tests/

scripts/:
lib.sh
prune-merged-worktrees.sh
tests
verify-pr-labels.sh
wait-for-review.sh

scripts/tes

ts/:
run.sh
test-lib.sh


### 1.2 — Why factor pure logic into `lib.sh`?

Because `gh`/`git` calls can't be unit-tested without network/auth — but the **decisions**
(set-difference of labels, "is this review newer than the commit", parsing worktrees) are pure
string processing. We isolate those so they're testable. Here are the three functions:

In [4]:
%%bash
cd "$ROOT"
grep -n '^[a-z_]*() {' scripts/lib.sh

12:label_diff() {
24:is_review_fresh() {
36:parse_worktrees() {


### 1.3 — `label_diff` — the silent-no-op label gotcha

`gh pr create --label X` *silently* drops labels the repo doesn't have at create time, so PR
labels go missing. `label_diff` computes which of the issue's labels are absent from the PR so we
can re-add them. Note it handles **labels containing spaces/colons** (`ai-tool: claude-code`) —
a classic place naive `for` loops break.

In [5]:
%%bash
cd "$ROOT"
source scripts/lib.sh

ISSUE_LABELS=$'type:bug\npriority:high\nai-tool: claude-code\ncomponent:api'
PR_LABELS=$'type:bug\ncomponent:api'

echo "Issue has:"; printf '  %s\n' "$ISSUE_LABELS"
echo "PR has:";    printf '  %s\n' "$PR_LABELS"
echo
echo "--> label_diff says these are MISSING from the PR:"
label_diff "$ISSUE_LABELS" "$PR_LABELS" | sed 's/^/  /'

Issue has:


  type:bug
priority:high
ai-tool: claude-code
component:api
PR has:
  type:bug
component:api

--> la

bel_diff says these are MISSING from the PR:


  ai-tool: claude-code
  priority:high


### 1.4 — `is_review_fresh` — avoiding the `head_sha` gotcha

The old loop polled *workflow runs by SHA*, which silently reports the merge commit, not the PR
head — so it never broke. Instead we poll the PR's reviews directly and ask: *is the latest review
newer than the latest commit?* ISO-8601 timestamps in the same zone compare lexicographically, so
no date parsing needed.

In [6]:
%%bash
cd "$ROOT"
source scripts/lib.sh

check () {  # $1=review ts, $2=commit ts
  if is_review_fresh "$1" "$2"; then echo "FRESH   (review $1 > commit $2)"
  else echo "not yet (review '$1' <= commit $2)"; fi
}
check "2026-06-30T10:00:00Z" "2026-06-30T09:00:00Z"   # review after commit
check "2026-06-30T08:00:00Z" "2026-06-30T09:00:00Z"   # review before commit (stale)
check ""                      "2026-06-30T09:00:00Z"   # no review yet

FRESH   (review 2026-06-30T10:00:00Z > commit 2026-06-30T09:00:00Z)


not yet (review '2026-06-30T08:00:00Z' <= commit 2026-06-30T09:00:00Z)
not yet (review '' <= commit 

2026-06-30T09:00:00Z)


### 1.5 — `parse_worktrees` — safe cleanup parsing

Cleanup must only ever touch the plugin's *own* worktrees. This parses
`git worktree list --porcelain` into `path<TAB>branch`, **dropping the main worktree** (always
first) and any **detached-HEAD** entries.

In [7]:
%%bash
cd "$ROOT"
source scripts/lib.sh

SAMPLE='worktree /home/me/repo
HEAD aaaa
branch refs/heads/main

worktree /home/me/repo-wt-42
HEAD bbbb
branch refs/heads/feat/42-thing

worktree /home/me/detached-one
HEAD cccc
detached
'
echo "parsed (main + detached correctly dropped):"
parse_worktrees "$SAMPLE" | sed 's/^/  /'

parsed (main + detached correctly dropped):


  /home/me/repo-wt-42	feat/42-thing


### 1.6 — The wrapper scripts: arg validation & safety

The three executables wrap that pure logic with the `gh`/`git` I/O. They validate args and fail
loudly (exit `64` = usage error) — important because they run **unattended**.

In [8]:
%%bash
cd "$ROOT"
./scripts/verify-pr-labels.sh   # no args on purpose
echo "exit code: $?  (64 = EX_USAGE — refuses to run without <pr> <issue>)"

usage: verify-pr-labels.sh <pr-number> <issue-number>


exit code: 64  (64 = EX_USAGE — refuses to run without <pr> <issue>)


### 1.7 — `prune-merged-worktrees.sh --dry-run` (read-only, safe)

This is the *real* script running against this repo. Watch the **safety gate**: it only considers
worktrees whose name matches `*-wt-*`. Our actual worktrees here are named `design-docs`,
`phase-1`, `phase-2` — so they're all **kept**, never touched. (`--dry-run` changes nothing.)

In [9]:
%%bash
cd "$ROOT"
./scripts/prune-merged-worktrees.sh --dry-run

summary: removed=0 kept=3 (dry-run)


### 1.8 — Isolation detection (the "stop owning isolation" change)

`ship_issue` no longer blindly creates a worktree — under Agent View / `--bg --worktree` the
harness already isolated us, and nesting a worktree breaks things. This one-liner detects it.
We're *inside* a linked worktree right now, so it should report **ISOLATED=1**.

In [10]:
%%bash
cd "$ROOT"
if [ "$(git rev-parse --git-common-dir)" != "$(git rev-parse --git-dir)" ]; then
  ISOLATED=1
else
  ISOLATED=0
fi
echo "git-dir       : $(git rev-parse --git-dir)"
echo "git-common-dir: $(git rev-parse --git-common-dir)"
echo "ISOLATED=$ISOLATED   -> ship_issue will NOT create a nested worktree (uses this one)"

git-dir       : /home/matt/projects/agentic-sdlc/.git/worktrees/phase-2


git-common-dir: /home/matt/projects/agentic-sdlc/.git


ISOLATED=1   -> ship_issue will NOT create a nested worktree (uses this one)


### 1.9 — The test suite (what CI/you would run)

In [11]:
%%bash
cd "$ROOT"
bash scripts/tests/run.sh

syntax OK: /home/matt/projects/agentic-sdlc/.claude/worktrees/phase-2/scripts/lib.sh


syntax OK: /home/matt/projects/agentic-sdlc/.claude/worktrees/phase-2/scripts/prune-merged-worktrees

.sh


syntax OK: /home/matt/projects/agentic-sdlc/.claude/worktrees/phase-2/scripts/verify-pr-labels.sh


syntax OK: /home/matt/projects/agentic-sdlc/.claude/worktrees/phase-2/scripts/wait-for-review.sh


syntax OK: /home/matt/projects/agentic-sdlc/.claude/worktrees/phase-2/scripts/tests/run.sh


syntax OK: /home/matt/projects/agentic-sdlc/.claude/worktrees/phase-2/scripts/tests/test-lib.sh


test-lib: 12 passed, 0 failed


unit tests OK
----------------------------------------
ALL PASS


---
# Part 2 — Phase 2: the two-tier knowledge model

Phase 1 was plumbing. Phase 2 is the **moat**: how the system *learns and corrects*. Two tiers of
knowledge with different authority, kept lean by an admission bar, fed by the compounding loop.

### 2.1 — The model (`KNOWLEDGE.md`)

In [12]:
%%bash
cd "$ROOT"
sed -n '/## Two tiers/,/## The admission bar/p' KNOWLEDGE.md

## Two tiers, two authorities

| Tier | What it is | Example | Authority |
| --- | --- | --- | --- |


| **Invariants** (taste / convention) | Arbitrary-but-consistent project choices | "Use Pydantic, n

ot dataclasses" | **Enforced** — a PR must not violate them. |
| **Constitution** (best-practice p

rinciples) | Objective-ish engineering principles | "No error handling for impossible states" | **Ju

stify-or-deviate** — a violation is *flagged*; you must consciously justify or fix. **Present evid

ence always beats the stored rule** — never a silent override. |

## The admission bar


### 2.2 — The constitution seed (`templates/constitution.md`)

The "correct me, don't just mirror me" half. **Principles, not conventions** — a violation is a
flag to *justify-or-deviate*, never a silent override. Host repos install and **curate** it.

In [13]:
%%bash
cd "$ROOT"
sed -n '1,26p' templates/constitution.md

# Constitution

These are best-practice **principles, not conventions**. A principle is a flag
to **

justify-or-deviate**: present evidence beats the rule, but the deviation
must be **conscious**, neve

r silent. Conventions and taste (arbitrary-but-
consistent project choices) live in the **invariants

** section, not here.

**Install:** drop this into the host repo's `AGENTS.md`, or keep it as
`const

itution.md` imported into it. Then **curate it** — strike any principle
you'd never actually enfor

...

ake illegal states unrepresentable — types/enums over stringly-typed flags.
- Depend on interfaces

 you own, not on incidental details of what you call.

## Code

- Don't handle errors that can't hap

pen; let impossible states crash loudly.
- Validate at trust boundaries only; trust your own interna

ls.
- Comments say WHY, never WHAT; delete commented-out code.
- Isolate side effects at the edges; 

keep the core pure where practical.
- Name for the reader who lacks your context.


### 2.3 — The compounding-loop decision flow, made runnable

`ship_issue` §12d is prose, but it's a deterministic algorithm. Here it is as Python so we can
**watch it route findings** the way the loop will: central-judge legitimacy → admission bar →
route by tier. This is the logic you're approving when you read §12.

In [14]:
from dataclasses import dataclass

@dataclass
class Finding:
    desc: str
    legitimate: bool          # central judging: is the reviewer actually right?
    already_a_rule: bool      # maps to an existing invariant/principle/ADR?
    catches_real_mistake: bool  # the admission bar
    kind: str                 # 'taste' | 'principle' | 'architectural'

def classify(f: Finding) -> str:
    if not f.legitimate:
        return "REJECT — central judging: confidently-wrong; dismiss inline, fold nothing in"
    if f.already_a_rule:
        return "SKIP — already covered (the system worked); log which rule caught it"
    if not f.catches_real_mistake:
        return "SKIP — fails the admission bar (wouldn't catch a real mistake)"
    return {
        "taste":         "ADD INVARIANT (enforced) -> AGENTS.md  [docs(invariants):]",
        "principle":     "ADD CONSTITUTION (justify-or-deviate) -> AGENTS.md  [docs(constitution):]",
        "architectural": "WRITE ADR -> docs/decisions/  [docs(adr):]",
    }[f.kind]

findings = [
    Finding("Use Pydantic BaseModel, not a raw dict, for the new response", True, False, True, "taste"),
    Finding("This try/except can never fire — impossible state", True, False, True, "principle"),
    Finding("Rename `x` to `data` (style nit)", True, False, False, "taste"),
    Finding("Re-validates a framework guarantee", True, True, True, "principle"),
    Finding("Bot insists on a 'fix' that would break the test", False, False, True, "principle"),
    Finding("Switch the whole service to event-sourcing", True, False, True, "architectural"),
]

w = max(len(f.desc) for f in findings)
for f in findings:
    print(f"- {f.desc:<{w}}  ->  {classify(f)}")

- Use Pydantic BaseModel, not a raw dict, for the new response  ->  ADD INVARIANT (enforced) -> AGENTS.md  [docs(invariants):]
- This try/except can never fire — impossible state             ->  ADD CONSTITUTION (justify-or-deviate) -> AGENTS.md  [docs(constitution):]
- Rename `x` to `data` (style nit)                              ->  SKIP — fails the admission bar (wouldn't catch a real mistake)
- Re-validates a framework guarantee                            ->  SKIP — already covered (the system worked); log which rule caught it
- Bot insists on a 'fix' that would break the test              ->  REJECT — central judging: confidently-wrong; dismiss inline, fold nothing in
- Switch the whole service to event-sourcing                    ->  WRITE ADR -> docs/decisions/  [docs(adr):]


### 2.4 — …and that maps directly to the prose in `ship_issue.md` §12d

In [15]:
%%bash
cd "$ROOT"
sed -n '/### 12d\./,/### 12e\./p' commands/ship_issue.md

### 12d. Judge, then classify each finding

**First, central-judge legitimacy.** Before harvesting a

nything, decide whether each finding is *actually correct* — reviewers (bot or human) are sometime

s confidently wrong. Reject illegitimate findings (dismiss inline; never fold an incorrect rule in).

 Only legitimate findings proceed.

**Then classify — be deterministic, don't invent rules to fill

 the step:**

1. **APPROVE + zero inline comments, or only style nits / praise** → skip. Note "no 

...

 add it. Then route by tier:
   - **Taste / convention** (this project's arbitrary-but-consistent ch

oice) → a one-line **invariant** (enforced).
   - **Best-practice principle** (objective engineeri

ng rule) → a one-line **constitution** principle (justify-or-deviate). Seed from `templates/consti

tution.md` if the repo has none yet.
   - **Architecturally significant** → an ADR instead. Most f

indings are invariant- or principle-level.

### 12e. Commit the addition


---
## Wrap-up — what you're signing off on

**Phase 1 (#8)** — the pipeline is now *fire-and-forget-ready*:
- Pure logic in `lib.sh` is unit-tested (12/12); the `gh`/`git` wrappers add I/O + arg-validation.
- `ship_issue` detects existing isolation instead of nesting worktrees, and never prompts.
- The prune safety gate (`*-wt-*`) means cleanup can't touch worktrees it didn't create.

**Phase 2 (#9)** — the knowledge layer *corrects* as well as learns:
- Two tiers: **invariants** (enforced) vs **constitution** (justify-or-deviate, evidence beats the rule).
- The loop **central-judges** findings and applies an **admission bar** before anything is written — so it can't bloat or fold in a wrong rule.

**Still open / tracked:** constitution pruning (#4), park-zone tuning (#5), AFK safety env (#6),
and Phase 3 (decompose `ship_issue` into `SKILL.md` skills).